(user-foundations-performance-benchmarks)=
# Benchmarks

Empirical performance metrics and throughput comparisons demonstrate MolSysMT's scaling efficiency and comparative execution speed against industry-standard libraries (**MDTraj**, **MDAnalysis**, and **SciPy**).

## Competitive Performance Timing Matrix

The table below presents comparative median execution durations across standard structural biology operations (evaluated on Trp-Cage miniprotein `1l2y` and Chicken Villin HP35 datasets):

In [1]:
import pandas as pd
from IPython.display import HTML

data = [
    {"Operation Area": "Trajectory Load (DCD)", "MolSysMT Public": "176.95 ms", "MolSysMT Native Kernel": "N/A (Streaming)", "MDTraj": "29.65 ms", "MDAnalysis": "49.11 ms"},
    {"Operation Area": "Selection Simple (CA)", "MolSysMT Public": "5.83 ms", "MolSysMT Native Kernel": "N/A", "MDTraj": "3.95 ms", "MDAnalysis": "0.21 ms"},
    {"Operation Area": "Selection Complex", "MolSysMT Public": "10.54 ms", "MolSysMT Native Kernel": "N/A", "MDTraj": "58.33 ms", "MDAnalysis": "0.70 ms"},
    {"Operation Area": "Center of Geometry", "MolSysMT Public": "291.13 ms", "MolSysMT Native Kernel": "9.38 ms", "MDTraj": "1.79 ms", "MDAnalysis": "193.22 ms"},
    {"Operation Area": "RMSD Calculation", "MolSysMT Public": "306.49 ms", "MolSysMT Native Kernel": "8.34 ms", "MDTraj": "0.65 ms", "MDAnalysis": "177.92 ms"},
    {"Operation Area": "Pairwise Distances", "MolSysMT Public": "640,028 ms", "MolSysMT Native Kernel": "24,324 ms", "MDTraj": "4,649 ms (SciPy)", "MDAnalysis": "3,496 ms"}
]

df = pd.DataFrame(data)
html_table = df.to_html(classes="table", index=False)
html_table = html_table.replace('<th', '<th style="text-align: left;"').replace('<td', '<td style="text-align: left;"')
HTML(html_table)

Operation Area,MolSysMT Public,MolSysMT Native Kernel,MDTraj,MDAnalysis
Trajectory Load (DCD),176.95 ms,N/A (Streaming),29.65 ms,49.11 ms
Selection Simple (CA),5.83 ms,N/A,3.95 ms,0.21 ms
Selection Complex,10.54 ms,N/A,58.33 ms,0.70 ms
Center of Geometry,291.13 ms,9.38 ms,1.79 ms,193.22 ms
RMSD Calculation,306.49 ms,8.34 ms,0.65 ms,177.92 ms
Pairwise Distances,"640,028 ms","24,324 ms","4,649 ms (SciPy)","3,496 ms"


## Key Architectural Observations

- **Complex Selections**: MolSysMT's selection parser resolves complex Boolean queries (`(name CA or name CB) and resname ALA VAL LEU`) in **10.5 ms**, outperforming MDTraj's bytecode compilation by **5.5x**.
- **Native Kernel Acceleration**: Calling MolSysMT's native compiled kernels directly achieves **~20x faster** Center of Geometry and **~21x faster** RMSD calculations compared to MDAnalysis wrappers.
- **Public API Digestion Tax**: The public API performs input validation, physical unit enforcement (`pyunitwizard`), and periodic boundary digestion. Controlled internal delegation may use `skip_digestion=True` only after the complete callee contract is established; MolSysMT has no value-passport protocol.